# Processing gnomAD and ClinVar data

In [ ]:
import os
import subprocess
import polars as pl
import pandas as pd
import re

# Set strict error handling
def run_command(cmd):
    result = subprocess.run(cmd, shell=True, check=True, text=True)
    return result

## ClinVar

ClinVar variants released on 2024-03-01 and the data is accessed on March 13, 2025

### Clean up ClinVar annotations

For merging the ClinVar with the VarChAMP variants more easily

In [ ]:
dest_path = "../1_inputs/raw_inputs/clinvar/variant_summary.txt.gz"
## Define the new processed and cleaned clinvar file
processed_clinvar_parq = "../3_outputs/processed_data/clinvar/clinvar_processed.parquet"

In [ ]:
## Read CSV with Polars (auto-decompresses .gz)
clinvar = pl.read_csv(dest_path, separator='\t', infer_schema_length=1000000)

## Process data with Polars expressions
clinvar = (
    clinvar
    .filter(pl.col("Assembly") == "GRCh38")
    .with_columns(
        pl.col("Name")
        # Split the string on '.', '(' or ')'
        .map_elements(lambda s: re.split(r"[.\(\)]", s), pl.List(pl.String))
        .alias("Name_listed")
    )
)
clinvar = clinvar.with_columns(
    pl.col("Name_listed")
    .list.to_struct(
            fields=[
            'NM', 'version', 'gene', 
            'c', 'clinvar_nt_change', 
            'p', 'clinvar_aa_change'
            ],
            upper_bound=7)
    .struct.unnest()
).with_columns(
        SPDI=pl.col("ChromosomeAccession") + ":" + 
            pl.col("PositionVCF").cast(str) + ":" +
            pl.col("ReferenceAlleleVCF") + ":" +
            pl.col("AlternateAlleleVCF"),
        RefSeq_mRNA=pl.col("NM") + "." + pl.col("version"),
        StarStatus=pl.when(pl.col("ReviewStatus").str.contains(r'single|conflicting'))
            .then(1)
            .when(pl.col("ReviewStatus").str.contains('multiple'))
            .then(2)
            .when(pl.col("ReviewStatus").str.contains('expert'))
            .then(3)
            .when(pl.col("ReviewStatus").str.contains('practice'))
            .then(4)
            .otherwise(None)
)

In [22]:
## Create the mapping expression with precise logic for a cleaner ClinVar Annotation
clinvar_mapping = (
    # First check for Conflicting interpretations (highest priority)
    pl.when(
        pl.col("ClinicalSignificance").str.to_lowercase().str.contains("conflicting")
    )
    .then(pl.lit("3_Conflicting"))
    
    # Then check for Uncertain significance (VUS)
    .when(pl.col("ClinicalSignificance").str.to_lowercase().str.contains("uncertain"))
    .then(pl.lit("4_VUS"))
    
    # # Special case for strings containing both "Uncertain significance" and "Pathogenic"
    # .when(
    #     pl.col("clinvar_clnsig").str.to_lowercase().str.contains("uncertain") &
    #     pl.col("clinvar_clnsig").str.to_lowercase().str.contains("pathogenic")
    # )
    # .then(pl.lit("4_VUS"))  # The dictionary says to classify this as VUS
    
    # Check for Pathogenic
    .when(pl.col("ClinicalSignificance").str.to_lowercase().str.contains("pathogenic"))
    .then(pl.lit("1_Pathogenic"))
    
    # Check for Benign
    .when(pl.col("ClinicalSignificance").str.to_lowercase().str.contains("benign"))
    .then(pl.lit("2_Benign"))
    
    # Everything else goes to Others
    .otherwise(pl.lit("5_Others"))
)

# Apply the mapping to create a new column
clinvar = clinvar.with_columns(clinvar_clnsig_clean=clinvar_mapping)

# Create a dataframe with unique mappings to check accuracy
unique_mappings = clinvar.select(
    pl.col("ClinicalSignificance"),
    pl.col("clinvar_clnsig_clean")
).unique()

# Sort by original value for easier verification
unique_mappings = unique_mappings.sort("ClinicalSignificance")

# Print each mapping to verify accuracy
print("Original Value -> Mapped Value")
print("-" * 50)
for row in unique_mappings.rows():
    print(f"{row[0]} -> {row[1]}")

Original Value -> Mapped Value
--------------------------------------------------
- -> 5_Others
Affects -> 5_Others
Affects; risk factor -> 5_Others
Benign -> 2_Benign
Benign/Likely benign -> 2_Benign
Benign/Likely benign; association -> 2_Benign
Benign/Likely benign; drug response -> 2_Benign
Benign/Likely benign; drug response; other -> 2_Benign
Benign/Likely benign; other -> 2_Benign
Benign/Likely benign; other; risk factor -> 2_Benign
Benign/Likely benign; risk factor -> 2_Benign
Benign; Affects -> 2_Benign
Benign; Affects; association; other -> 2_Benign
Benign; association -> 2_Benign
Benign; confers sensitivity -> 2_Benign
Benign; drug response -> 2_Benign
Benign; other -> 2_Benign
Benign; protective -> 2_Benign
Benign; risk factor -> 2_Benign
Conflicting classifications of pathogenicity -> 3_Conflicting
Conflicting classifications of pathogenicity; Affects -> 3_Conflicting
Conflicting classifications of pathogenicity; association -> 3_Conflicting
Conflicting classifications of p

In [ ]:
# ## Make sure we only drop columns that actually exist
# columns_to_drop = [col for col in ['NM', 'version', 'gene', 'c', 'p', 'ReferenceAllele', 'AlternateAllele', 'Name_listed'] 
#                   if col in clinvar.columns]
# if columns_to_drop:
#     clinvar = clinvar.drop(columns_to_drop)

# # Now cast to categorical
# clinvar = clinvar.with_columns(
#     pl.col(["Assembly", "ClinicalSignificance", "ReviewStatus", "clinvar_clnsig_clean"]).cast(pl.Categorical)
# )

# # Write to Parquet with ZSTD compression
# clinvar.write_parquet(processed_clinvar_parq, 
#                      compression="zstd", 
#                      statistics=True)

## Merge clinvar and gnomad to varchamp alleles

### Loading varchamp alleles

In [ ]:
## get unique info for genes in polars
gene_info = pl.scan_csv(f"../1_inputs/VarChamp_Consolidated_allele_collection.tsv", separator="\t", infer_schema_length=100000) #[gene_cols]
gene_info.collect()

## Loading for pandas version for smaller and easier processing
gene_info_pd = pd.read_csv(f"../1_inputs/VarChamp_Consolidated_allele_collection.tsv", sep="\t")
gene_info_pd["chrom"] = "chr" + gene_info_pd["chr"]
gene_info_pd["ref"] = gene_info_pd["nt_change"].apply(lambda x: x.split('>')[0][-1])
gene_info_pd["alt"] = gene_info_pd["nt_change"].apply(lambda x: x.split('>')[-1][0])

### Merge with ClinVar

In [36]:
processed_clinvar_parq = "../3_outputs/processed_data/clinvar/clinvar_processed.parquet"
gene_info_clinvar_parq = "../3_outputs/varchamp_consolidated_alleles_clinvar_annot.parquet"
gene_info_clinvar_tsv = "../3_outputs/varchamp_consolidated_alleles_clinvar_annot.tsv"

clinvar = pl.scan_parquet(processed_clinvar_parq)
clinvar.collect().head()


# gene_info_clinvar = gene_info.join(
#                         clinvar,
#                         left_on="spdi",
#                         right_on="SPDI",
#                         how="left"
#                     )
# gene_info_clinvar = gene_info_clinvar.collect()

# gene_info_clinvar.write_parquet(gene_info_clinvar_parq, compression="zstd")
# gene_info_clinvar.write_csv(gene_info_clinvar_tsv, separator="\t")

#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,LastEvaluated,RS# (dbSNP),nsv/esv (dbVar),RCVaccession,PhenotypeIDS,PhenotypeList,Origin,OriginSimple,Assembly,ChromosomeAccession,Chromosome,Start,Stop,Cytogenetic,ReviewStatus,NumberSubmitters,Guidelines,TestedInGTR,OtherIDs,SubmitterCategories,VariationID,PositionVCF,ReferenceAlleleVCF,AlternateAlleleVCF,SomaticClinicalImpact,SomaticClinicalImpactLastEvaluated,ReviewStatusClinicalImpact,Oncogenicity,OncogenicityLastEvaluated,ReviewStatusOncogenicity,SCVsForAggregateGermlineClassification,SCVsForAggregateSomaticClinicalImpact,SCVsForAggregateOncogenicityClassification,clinvar_nt_change,clinvar_aa_change,SPDI,RefSeq_mRNA,StarStatus,clinvar_clnsig_clean
i64,str,str,i64,str,str,cat,i64,str,i64,str,str,str,str,str,str,cat,str,str,i64,i64,str,cat,i64,str,str,str,i64,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i32,cat
15041,"""Indel""","""NM_014855.3(AP5Z1):c.80_83deli…",9907,"""AP5Z1""","""HGNC:22197""","""Pathogenic""",1,"""Jun 25, 2024""",397704705,"""-""","""RCV000000012|RCV004998069""","""MONDO:MONDO:0013342,MedGen:C31…","""Hereditary spastic paraplegia …","""germline;unknown""","""germline""","""GRCh38""","""NC_000007.14""","""7""",4781213,4781216,"""7p22.1""","""criteria provided, multiple su…",3,"""-""","""N""","""ClinGen:CA215070,OMIM:613653.0…",3,2,4781213,"""GGAT""","""TGCTGTAAACTGTAACTGTAAA""","""-""","""-""","""-""","""-""","""-""","""-""","""SCV001451119|SCV005622007""","""-""","""-""","""80_83delinsTGCTGTAAACTGTAACTGT…","""Arg27_Ile28delinsLeuLeuTer""","""NC_000007.14:4781213:GGAT:TGCT…","""NM_014855.3""",2,"""1_Pathogenic"""
15042,"""Deletion""","""NM_014855.3(AP5Z1):c.1413_1426…",9907,"""AP5Z1""","""HGNC:22197""","""Pathogenic""",1,"""Jun 29, 2010""",397704709,"""-""","""RCV000000013""","""MONDO:MONDO:0013342,MedGen:C31…","""Hereditary spastic paraplegia …","""germline""","""germline""","""GRCh38""","""NC_000007.14""","""7""",4787730,4787743,"""7p22.1""","""no assertion criteria provided""",1,"""-""","""N""","""OMIM:613653.0002,ClinGen:CA215…",1,3,4787729,"""GCTGCTGGACCTGCC""","""G""","""-""","""-""","""-""","""-""","""-""","""-""","""SCV000020156""","""-""","""-""","""1413_1426del ""","""Leu473fs""","""NC_000007.14:4787729:GCTGCTGGA…","""NM_014855.3""",null,"""1_Pathogenic"""
15043,"""single nucleotide variant""","""NM_014630.3(ZNF592):c.3136G>A …",9640,"""ZNF592""","""HGNC:28986""","""Uncertain significance""",0,"""Jun 29, 2015""",150829393,"""-""","""RCV000000014""","""MONDO:MONDO:0033005,MedGen:C45…","""Galloway-Mowat syndrome 1""","""germline""","""germline""","""GRCh38""","""NC_000015.10""","""15""",84799209,84799209,"""15q25.3""","""no assertion criteria provided""",1,"""-""","""N""","""ClinGen:CA210674,UniProtKB:Q92…",1,4,84799209,"""G""","""A""","""-""","""-""","""-""","""-""","""-""","""-""","""SCV000020157""","""-""","""-""","""3136G>A ""","""Gly1046Arg""","""NC_000015.10:84799209:G:A""","""NM_014630.3""",null,"""4_VUS"""
15044,"""single nucleotide variant""","""NM_017547.4(FOXRED1):c.694C>T …",55572,"""FOXRED1""","""HGNC:26927""","""Pathogenic""",1,"""Mar 12, 2024""",267606829,"""-""","""RCV000000015|RCV000578659|RCV0…","""MONDO:MONDO:0032624,MedGen:C47…","""Mitochondrial complex 1 defici…","""germline""","""germline""","""GRCh38""","""NC_000011.10""","""11""",126275389,126275389,"""11q24.2""","""criteria provided, multiple su…",6,"""-""","""N""","""ClinGen:CA113792,OMIM:613622.0…",3,5,126275389,"""C""","""T""","""-""","""-""","""-""","""-""","""-""","""-""","""SCV000680696|SCV001363290|SCV0…","""-""","""-""","""694C>T ""","""Gln232Ter""","""NC_000011.10:126275389:C:T""","""NM_017547.4""",2,"""1_Pathogenic"""
15045,"""single nucleotide variant""","""NM_017547.4(FOXRED1):c.1289A>G…",55572,"""FOXRED1""","""HGNC:26927""","""Likely pathogenic""",1,"""Jun 06, 2024""",267606830,"""-""","""RCV000000016""","""MONDO:MONDO:0032624,MedGen:C47…","""Mitochondrial complex 1 defici…","""germline""","""germline""",

In [38]:
clinvar.collect().shape

(3390885, 47)

#### Report the number of alleles missing ClinVar annotations

In [27]:
gene_info_clinvar = pl.scan_parquet(gene_info_clinvar_parq)
gene_info_clinvar.filter((~pl.col("spdi").is_null())&(pl.col("#AlleleID").is_null())).collect()

orf_id,symbol,nt_change,aa_change,ensembl_gene_id,chr,chr_pos_38,spdi,ccsb_allele_id,ccsb_mutation_id,mutation_id_rc4,mutation_id_cegs2,mutation_id_edgotyping3,entry_plate,entry_well,collection,sequence,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,LastEvaluated,RS# (dbSNP),nsv/esv (dbVar),RCVaccession,PhenotypeIDS,PhenotypeList,Origin,OriginSimple,Assembly,ChromosomeAccession,Chromosome,Start,Stop,Cytogenetic,ReviewStatus,NumberSubmitters,Guidelines,TestedInGTR,OtherIDs,SubmitterCategories,VariationID,PositionVCF,ReferenceAlleleVCF,AlternateAlleleVCF,SomaticClinicalImpact,SomaticClinicalImpactLastEvaluated,ReviewStatusClinicalImpact,Oncogenicity,OncogenicityLastEvaluated,ReviewStatusOncogenicity,SCVsForAggregateGermlineClassification,SCVsForAggregateSomaticClinicalImpact,SCVsForAggregateOncogenicityClassification,clinvar_nt_change,clinvar_aa_change,RefSeq_mRNA,StarStatus,clinvar_clnsig_clean
i64,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,cat,i64,str,i64,str,str,str,str,str,str,cat,str,str,i64,i64,str,cat,i64,str,str,str,i64,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i32,cat
2,"""GBA""","""160G>C""","""Val54Leu""","""ENSG00000177628""","""1""",155240033,"""NC_000001.11:155240033:C:G""","""ALE0000584""","""CCSBVarC000001""","""6""","""NULL""","""NULL""","""GDEhDisVCh_40054""","""F12""","""RC4""","""ATGGAGTTTTCAAGTCCTTCCAGAGAGGAA…",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2,"""GBA""","""670C>T""","""Leu224Phe""","""ENSG00000177628""","""1""",155238225,"""NC_000001.11:155238225:G:A""","""ALE00000002""","""CCSBVarC000002""","""73""","""NULL""","""NULL""","""GDEhDisVCh_40054""","""C12""","""RC4""","""ATGGAGTTTTCAAGTCCTTCCAGAGAGGAA…",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
167,"""ACTA2""","""874A>G""","""Arg292Gly""","""ENSG00000107796""","""10""",88938177,"""NC_000010.11:88938177:T:C""","""ALE00000012""","""CCSBVarC000012""","""390""","""NULL""","""NULL""","""GDEhDisVCh_40054""","""B12""","""RC4""","""ATGTGTGAAGAAGAGGACAGCACTGCCTTG…",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
187,"""FOXA3""","""271G>A""","""Gly91Arg""","""ENSG00000170608""","""19""",45872276,"""NC_000019.10:45872276:G:A""","""ALE00000013""","""CCSBVarC000013""","""393""","""NULL""","""NULL""","""GDEhDisVCh_40008""","""G09""","""RC4""","""ATGCTGGGCTCAGTGAAGATGGAGGCCCAT…",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
286,"""ABHD5""","""245A>G""","""His82Arg""","""ENSG00000011198""","""3""",43702326,"""NC_000003.12:43702326:A:G""","""ALE00000031""","""CCSBVarC000031""","""782""","""NULL""","""NULL""","""GDEhDisVCh_40077""","""H08""","""RC4""","""ATGGCGGCGGAGGAGGAGGAGGTGGACTCT…",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
4765,"""ITGB2""","""769C>T""","""Arg257Trp""","""ENSG00000160255""","""21""",44900448,"""NC_000021.9:44900448:G:A""","""ALE00008408""","""CCSBVarC008422""","""NULL""","""100749""","""NULL""","""GDEhDisVCh_40016

Check why missing, and it seems like some alleles just weren't collected by ClinVar yet

In [29]:
gene_info_clinvar.filter(pl.col("symbol")=="GBA").sort(by="clinvar_aa_change").collect()

orf_id,symbol,nt_change,aa_change,ensembl_gene_id,chr,chr_pos_38,spdi,ccsb_allele_id,ccsb_mutation_id,mutation_id_rc4,mutation_id_cegs2,mutation_id_edgotyping3,entry_plate,entry_well,collection,sequence,#AlleleID,Type,Name,GeneID,GeneSymbol,HGNC_ID,ClinicalSignificance,ClinSigSimple,LastEvaluated,RS# (dbSNP),nsv/esv (dbVar),RCVaccession,PhenotypeIDS,PhenotypeList,Origin,OriginSimple,Assembly,ChromosomeAccession,Chromosome,Start,Stop,Cytogenetic,ReviewStatus,NumberSubmitters,Guidelines,TestedInGTR,OtherIDs,SubmitterCategories,VariationID,PositionVCF,ReferenceAlleleVCF,AlternateAlleleVCF,SomaticClinicalImpact,SomaticClinicalImpactLastEvaluated,ReviewStatusClinicalImpact,Oncogenicity,OncogenicityLastEvaluated,ReviewStatusOncogenicity,SCVsForAggregateGermlineClassification,SCVsForAggregateSomaticClinicalImpact,SCVsForAggregateOncogenicityClassification,clinvar_nt_change,clinvar_aa_change,RefSeq_mRNA,StarStatus,clinvar_clnsig_clean
i64,str,str,str,str,str,i64,str,str,str,str,str,str,str,str,str,str,i64,str,str,i64,str,str,cat,i64,str,i64,str,str,str,str,str,str,cat,str,str,i64,i64,str,cat,i64,str,str,str,i64,i64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i32,cat
2,"""GBA""","""160G>C""","""Val54Leu""","""ENSG00000177628""","""1""",155240033,"""NC_000001.11:155240033:C:G""","""ALE0000584""","""CCSBVarC000001""","""6""","""NULL""","""NULL""","""GDEhDisVCh_40054""","""F12""","""RC4""","""ATGGAGTTTTCAAGTCCTTCCAGAGAGGAA…",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2,"""GBA""","""670C>T""","""Leu224Phe""","""ENSG00000177628""","""1""",155238225,"""NC_000001.11:155238225:G:A""","""ALE00000002""","""CCSBVarC000002""","""73""","""NULL""","""NULL""","""GDEhDisVCh_40054""","""C12""","""RC4""","""ATGGAGTTTTCAAGTCCTTCCAGAGAGGAA…",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2,"""GBA""","""1483G>C""","""Ala495Pro""","""ENSG00000177628""","""1""",155235217,"""NC_000001.11:155235217:C:G""","""ALE00003889""","""CCSBVarC003889""","""NULL""","""213560""","""NULL""","""GDEhDisVCh_40054""","""G07""","""CEGS2""","""ATGGAGTTTTCAAGTCCTTCCAGAGAGGAA…",38384,"""single nucleotide variant""","""NM_000157.4(GBA1):c.1483G>C (p…",2629,"""GBA1""","""HGNC:4177""","""Conflicting classifications of…",1,"""Aug 20, 2024""",368060,"""-""","""RCV000079341|RCV003129772""","""MedGen:CN169374|MedGen:C366190…","""not specified|not provided""","""germline""","""germline""","""GRCh38""","""NC_000001.11""","""1""",155235217,155235217,"""1q22""","""criteria provided, conflicting…",5,"""-""","""N""","""ClinGen:CA146980,UniProtKB:P04…",2,93450,155235217,"""C""","""G""","""-""","""-""","""-""","""-""","""-""","""-""","""SCV002556287|SCV002756106|SCV0…","""-""","""-""","""1483G>C ""","""Ala495Pro""","""NM_000157.4""",1,"""3_Conflicting"""
2,"""GBA""","""476G>A""","""Arg159Gln""","""ENSG00000177628""","""1""",155238629,"""NC_000001.11:155238629:C:T""","""ALE00003870""","""CCSBVarC003870""","""NULL""","""213515""","""NULL""","""GDEhDisVCh_40054""","""E09""","""CEGS2""","""ATGGAGTTTTCAAGTCCTTCCAGAGAGGAA…",19330,"""single nucleotide variant""","""NM_000157.4(GBA1):c.476G>A (p.…",2629,"""GBA1""","""HGNC:4177""","""Pathogenic/Likely pathogenic""",1,"""Jun 24, 2024""",79653797,"""-""","""RCV000004518|RCV000004519|RCV0…","""MONDO:MONDO:0009265,MedGen:C19…","""Gaucher disease type I|Gaucher…","""germline""","""germline""","""GRCh38""","""NC_000001.11""","""1""",155238629,155238629,"""1q22""","""criteria provided, multiple su…",8,"""-""","""N""","""OMIM:606463.0004,ClinGen:CA253…",3,4291,155238629,"""C""","""T""","""-""","""-""","""-""","""-""","""-""","""-""","""SCV001423035|SCV001425323|SCV0…","""-""","""-"""

### Merge with gnomad

##### Sort out all the vcf files and all their position coordinates for looking up varchamp-related variants

In [ ]:
## Generate all the position coordinates for varchamp-related variants for quicker look-up

## Define some constants
VCF_DIR = "../1_inputs/raw_inputs/gnomad"
CHROMOSOMES = [str(c) for c in list(range(1, 23))] + ['X', 'Y']

# VARIANT_LIST = f"../3_outputs/processed_data/gnomad_allele_frequencies/gnomad_variants_list.txt"
# gene_info_pd[["chrom","chr_pos_38","ref","alt"]].to_csv(VARIANT_LIST, sep="\t", header=False, index=False)

In [ ]:
# # !conda install bcftools -c conda-forge -y ## install bcftools if you didn't

# OUTPUT_DIR = f"../3_outputs/processed_data/gnomad_allele_frequencies"
# os.makedirs(OUTPUT_DIR, exist_ok=True)

# # Process chromosomes
# for chrom in CHROMOSOMES:
#     vcf_file = f"{VCF_DIR}/gnomad.exomes.v4.1.sites.chr{chrom}.vcf.bgz"
#     csi_file = f"{vcf_file}.csi"
#     output_file = f"{OUTPUT_DIR}/chr{chrom}_allele_frequencies.txt"
    
#     if os.path.exists(vcf_file):
#         print(f"Processing chromosome {chrom}...")
        
#         if not os.path.exists(csi_file):
#             print(f"Indexing {vcf_file}")
#             run_command(f"bcftools index --threads 4 {vcf_file}")
        
#         print(f"Extracting frequencies for {chrom}")
#         view_cmd = f"bcftools view -R {VARIANT_LIST} {vcf_file}"
#         query_cmd = "bcftools query -f '%CHROM\\t%POS\\t%REF\\t%ALT\\t%AF\\n'"
        
#         with open(output_file, 'w') as f:
#             p1 = subprocess.Popen(view_cmd.split(), stdout=subprocess.PIPE)
#             p2 = subprocess.Popen(query_cmd.split(), stdin=p1.stdout, stdout=f)
#             p1.stdout.close()
#             p2.communicate()
            
#         print(f"Saved to {output_file}")
#     else:
#         print(f"Skipping chromosome {chrom} - file not found")

# print("Processing completed")

Processing chromosome 1...
Extracting frequencies for 1
Saved to ../3_outputs/processed_data/gnomad_allele_frequencies/chr1_allele_frequencies.txt
Processing chromosome 2...
Extracting frequencies for 2
Saved to ../3_outputs/processed_data/gnomad_allele_frequencies/chr2_allele_frequencies.txt
Processing chromosome 3...
Extracting frequencies for 3
Saved to ../3_outputs/processed_data/gnomad_allele_frequencies/chr3_allele_frequencies.txt
Processing chromosome 4...
Extracting frequencies for 4
Saved to ../3_outputs/processed_data/gnomad_allele_frequencies/chr4_allele_frequencies.txt
Processing chromosome 5...
Extracting frequencies for 5
Saved to ../3_outputs/processed_data/gnomad_allele_frequencies/chr5_allele_frequencies.txt
Processing chromosome 6...
Extracting frequencies for 6
Saved to ../3_outputs/processed_data/gnomad_allele_frequencies/chr6_allele_frequencies.txt
Processing chromosome 7...
Extracting frequencies for 7
Saved to ../3_outputs/processed_data/gnomad_allele_frequencies

#### Merge the gnomad AF files from chromosomes

In [31]:
allele_freq_dfs = []
for chrom in CHROMOSOMES:
    try:
        allele_freq = pl.read_csv(f"../3_outputs/processed_data/gnomad_allele_frequencies/chr{chrom}_allele_frequencies.txt",
                                separator="\t", has_header=False, 
                                new_columns=["chr","chr_pos_38","ref","alt","allele_freq"])
        allele_freq = allele_freq.with_columns(
            pl.col("chr").str.replace(r"\'", "").str.replace(r"\'", "").str.replace("chr", "")
        ).drop_nulls()
        # allele_freq = allele_freq.filter(~pl.all_horizontal(pl.all().is_null()))
        # print(allele_freq)
        allele_freq_dfs.append(allele_freq)
    except:
        continue

allele_freq_all = pl.concat(allele_freq_dfs)
allele_freq_gnomad = allele_freq_all.with_columns(
    pl.concat_str([pl.col("chr"),
                   pl.lit('-'),
                   pl.col("chr_pos_38").cast(pl.String),
                   pl.lit('-'),
                   pl.col("ref"),
                   pl.lit('-'),
                   pl.col("alt")
                   ]).alias("varid"),
    pl.col("allele_freq").alias("gnomad_af")
).drop(pl.col("allele_freq")).to_pandas()

allele_freq_gnomad

,chr,chr_pos_38,ref,alt,varid,gnomad_af
0,1,2028259,C,A,1-2028259-C-A,6.850020e-07
1,1,2028259,C,T,1-2028259-C-T,1.409050e-03
2,1,2028260,G,A,1-2028260-G-A,2.174040e-02
3,1,5907193,CG,C,1-5907193-CG-C,0.000000e+00
4,1,5907194,G,A,1-5907194-G-A,1.325610e-05
...,...,...,...,...,...,...
9708,X,154863124,C,T,X-154863124-C-T,1.093600e-05
9709,X,154863125,G,A,X-154863125-G-A,3.645470e-06
9710,X,154863151,C,T,X-154863151-C-T,3.651830e-06
9711,X,155280059,G,C,X-155280059-G-C,1.160730e-04


### Merge the gnomad AF to VarChAMP alleles

In [32]:
gene_info_pd["varid"] = gene_info_pd["chr"] + "-" + gene_info_pd["chr_pos_38"].astype(str) + "-" + \
                        gene_info_pd["ref"] + "-" + gene_info_pd["alt"]

gene_info_gnomad = gene_info_pd.merge(allele_freq_gnomad,
                                      on="varid",
                                      how="left")

# gene_info_gnomad.to_csv("../3_outputs/varchamp_consolidated_alleles_gnomad_annot.tsv", sep="\t", index=False)

In [33]:
gene_info_gnomad

,orf_id,symbol,nt_change,aa_change,ensembl_gene_id,chr_x,chr_pos_38_x,spdi,ccsb_allele_id,ccsb_mutation_id,...,sequence,chrom,ref_x,alt_x,varid,chr_y,chr_pos_38_y,ref_y,alt_y,gnomad_af
0,2,GBA,160G>C,Val54Leu,ENSG00000177628,1,155240033,NC_000001.11:155240033:C:G,ALE0000584,CCSBVarC000001,...,ATGGAGTTTTCAAGTCCTTCCAGAGAGGAATGTCCCAAGCCTTTGA...,chr1,G,C,1-155240033-G-C,NaN,NaN,NaN,NaN,NaN
1,2,GBA,670C>T,Leu224Phe,ENSG00000177628,1,155238225,NC_000001.11:155238225:G:A,ALE00000002,CCSBVarC000002,...,ATGGAGTTTTCAAGTCCTTCCAGAGAGGAATGTCCCAAGCCTTTGA...,chr1,C,T,1-155238225-C-T,NaN,NaN,NaN,NaN,NaN
2,2,GBA,887G>A,Arg296Gln,ENSG00000177628,1,155237453,NC_000001.11:155237453:C:T,ALE00000003,CCSBVarC000003,...,ATGGAGTTTTCAAGTCCTTCCAGAGAGGAATGTCCCAAGCCTTTGA...,chr1,G,A,1-155237453-G-A,NaN,NaN,NaN,NaN,NaN
3,2,GBA,1448T>C,Leu483Pro,ENSG00000177628,1,155235252,NC_000001.11:155235252:A:G,ALE00000004,CCSBVarC000004,...,ATGGAGTTTTCAAGTCCTTCCAGAGAGGAATGTCCCAAGCCTTTGA...,chr1,T,C,1-155235252-T-C,NaN,NaN,NaN,NaN,NaN
4,25,SLC9A3R1,328C>G,Leu110Val,ENSG00000109062,17,74749174,NC_000017.11:74749174:C:G,ALE00000005,CCSBVarC000005,...,ATGAGCGCGGACGCAGCGGCCGGGGCGCCCCTGCCCCGGCTCTGCT...,chr17,C,G,17-74749174-C-G,17,74749174.0,C,G,0.026302
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8483,8999,TTR,355G>A,Asp119Asn,ENSG00000118271,18,31598586,NC_000018.10:31598586:G:A,ALE00008468,CCSBVarC008481,...,ATGGCTTCTCATCGTCTGCTCCTCCTCTGCCTTGCTGGACTGGTAT...,chr18,G,A,18-31598586-G-A,18,31598586.0,G,A,0.000079
8484,7323,PMP22,152A>G,His51Arg,ENSG00000109099,17,15259120,NC_000017.11:15259120:T:C,ALE00008469,CCSBVarC008466,...,ATGCTCCTCCTGTTGCTGAGTATCATCGTCCTCCACGTCGCGGTGC...,chr17,A,G,17-15259120-A-G,NaN,NaN,NaN,NaN,NaN
8485,2651,CLN6,89G>C,Gly30Ala,ENSG00000128973,15,68218645,NC_000015.10:68218645:C:G,ALE00008470,CCSBVarC008436,...,ATGGAGGCGACGCGGAGGCGGCAGCACCTGGGAGCGACGGGCGGCC...,chr15,G,C,15-68218645-G-C,NaN,NaN,NaN,NaN,NaN
8486,7323,PMP22,403G>A,Ala135Thr,ENSG00000109099,17,15230997,NC_000017.11:15230997:C:T,ALE00008471,CCSBVarC008467,...,ATGCTCCTCCTGTTGCTGAGTATCATCGTCCTCCACGTCGCGGTGC...,chr17,G,A,17-15230997-G-A,NaN,NaN,NaN,NaN,NaN


### Merge the ClinVar info AND gnomad AF BOTH to VarChAMP alleles

In [34]:
gene_info_clinvar_pd = gene_info_clinvar.collect().to_pandas()
gene_info_clinvar_pd["chrom"] = "chr" + gene_info_clinvar_pd["chr"]
gene_info_clinvar_pd["ref"] = gene_info_clinvar_pd["nt_change"].apply(lambda x: x.split('>')[0][-1])
gene_info_clinvar_pd["alt"] = gene_info_clinvar_pd["nt_change"].apply(lambda x: x.split('>')[-1][0])
gene_info_clinvar_pd["varid"] = gene_info_clinvar_pd["chr"] + "-" + gene_info_clinvar_pd["chr_pos_38"].astype(str) + "-" + \
                                gene_info_clinvar_pd["ref"] + "-" + gene_info_clinvar_pd["alt"]
gene_info_clinvar_gnomad_pd = gene_info_clinvar_pd.merge(allele_freq_gnomad[["varid","gnomad_af"]],
                                                  on="varid",how="left")

gene_info_clinvar_gnomad_pd.to_csv("../3_outputs/varchamp_consolidated_alleles_clinvar_gnomad.tsv", sep="\t", index=False)

In [35]:
# gene_info_clinvar_gnomad_pd
gene_info_clinvar_gnomad_pd[gene_info_clinvar_gnomad_pd["gnomad_af"].isna()]

,orf_id,symbol,nt_change,aa_change,ensembl_gene_id,chr,chr_pos_38,spdi,ccsb_allele_id,ccsb_mutation_id,...,clinvar_nt_change,clinvar_aa_change,RefSeq_mRNA,StarStatus,clinvar_clnsig_clean,chrom,ref,alt,varid,gnomad_af
0,2,GBA,160G>C,Val54Leu,ENSG00000177628,1,155240033,NC_000001.11:155240033:C:G,ALE0000584,CCSBVarC000001,...,None,None,None,NaN,NaN,chr1,G,C,1-155240033-G-C,NaN
1,2,GBA,670C>T,Leu224Phe,ENSG00000177628,1,155238225,NC_000001.11:155238225:G:A,ALE00000002,CCSBVarC000002,...,None,None,None,NaN,NaN,chr1,C,T,1-155238225-C-T,NaN
2,2,GBA,887G>A,Arg296Gln,ENSG00000177628,1,155237453,NC_000001.11:155237453:C:T,ALE00000003,CCSBVarC000003,...,887G>A,Arg296Gln,NM_000157.4,2.0,1_Pathogenic,chr1,G,A,1-155237453-G-A,NaN
3,2,GBA,1448T>C,Leu483Pro,ENSG00000177628,1,155235252,NC_000001.11:155235252:A:G,ALE00000004,CCSBVarC000004,...,1448T>C,Leu483Pro,NM_000157.4,2.0,1_Pathogenic,chr1,T,C,1-155235252-T-C,NaN
6,138,MYD88,278T>C,Leu93Pro,ENSG00000172936,3,38138978,NC_000003.12:38138978:T:C,ALE00000007,CCSBVarC000007,...,278T>C,Leu93Pro,NM_002468.5,NaN,1_Pathogenic,chr3,T,C,3-38138978-T-C,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8478,6666,MAP2K2,49A>G,Thr17Ala,ENSG00000126934,19,4123827,NC_000019.10:4123827:T:C,ALE00008463,CCSBVarC008461,...,49A>G,Thr17Ala,NM_030662.4,1.0,3_Conflicting,chr19,A,G,19-4123827-A-G,NaN
8484,7323,PMP22,152A>G,His51Arg,ENSG00000109099,17,15259120,NC_000017.11:15259120:T:C,ALE00008469,CCSBVarC008466,...,152A>G,His51Arg,NM_000304.4,1.0,3_Conflicting,chr17,A,G,17-15259120-A-G,NaN
8485,2651,CLN6,89G>C,Gly30Ala,ENSG00000128973,15,68218645,NC_000015.10:68218645:C:G,ALE00008470,CCSBVarC008436,...,89G>C,Gly30Ala,NM_017882.3,1.0,2_Benign,chr15,G,C,15-68218645-G-C,NaN
8486,7323,PMP22,403G>A,Ala135Thr,ENSG00000109099,17,15230997,NC_000017.11:15230997:C:T,ALE00008471,CCSBVarC008467,...,403G>A,Ala135Thr,NM_000304.4,2.0,2_Benign,chr17,G,A,17-15230997-G-A,NaN
